In [3]:
import psycopg2
import pandas as pd

# Shared connection parameters — change these to match your setup
DB_CONFIG = {
    "host":     "localhost",
    "database": "bank_reviews",
    "user":     "admin",
    "password": "admin1612"
}

In [ ]:
#create tables
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

# --- banks table ---
cur.execute("""
    CREATE TABLE IF NOT EXISTS banks (
        bank_id   SERIAL PRIMARY KEY,
        bank_name VARCHAR(255) UNIQUE,
        app_name VARCHAR(255) 
    );
""")

# --- reviews table  ---


cur.execute("""
    CREATE TABLE IF NOT EXISTS reviews (
        review_id     SERIAL PRIMARY KEY,
        bank_id INT REFERENCES banks(bank_id),
        review_text   TEXT,
        rating        INT,
        review_date   DATE,
        sentiment_label VARCHAR(20),
        sentiment_score FLOAT,
        identified_theme VARCHAR(100),
        source        VARCHAR(50)
    );
""")

conn.commit()
cur.close()
conn.close()

print("Tables created successfully.")

Tables created successfully.


In [17]:
# Read CSV files
df1 = pd.read_csv("../data/processed/dashen_bank_reviews_clean.csv")
df2 = pd.read_csv("../data/processed/boa_bank_reviews_clean.csv")
df3 = pd.read_csv("../data/processed/cbe_bank_reviews_clean.csv")

# Concatenate
combined_df = pd.concat([df1, df2, df3], ignore_index=True)

# Save result
combined_df.to_csv("combined_banks.csv", index=False)

df_banks = combined_df
df_reviews= pd.read_csv("../data/processed/combined_filtered_reviews.csv")
df = pd.concat([df_banks, df_reviews], axis=1)
df.columns
# df_reviews.columns

Index(['review', 'rating', 'date', 'bank', 'source', 'review_id',
       'review_text', 'sentiment_label', 'sentiment_score',
       'identified_theme'],
      dtype='str')

In [18]:
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()
for _, row in df.iterrows():

    cur.execute(
        """
        INSERT INTO banks (bank_name, app_name)
        VALUES (%s, %s)
        ON CONFLICT (bank_name) DO NOTHING;
        """,
        (
            row["bank"],
            row["source"]
        )
    )

conn.commit()

print("Banks inserted.")

cur.execute("SELECT bank_id, bank_name FROM banks")

bank_map = {
    bank_name.lower(): bank_id
    for bank_id, bank_name in cur.fetchall()
}

print(bank_map)
for _, row in df.iterrows():

    bank_id = bank_map.get(row["bank"])

    cur.execute(
        """
        INSERT INTO reviews
        (
            bank_id,
            review_text,
            rating,
            review_date,
            sentiment_label,
            sentiment_score,
            identified_theme,
            source
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s);
        """,
        (
            bank_id,
            row["review"],
            row["rating"],
            row["date"],
            row["sentiment_label"],
            row["sentiment_score"],
            row["identified_theme"],
            row["source"]
        )
    )

conn.commit()

print("Reviews inserted.")

Banks inserted.
{'dashen bank': 1, 'boa bank': 501, 'cbe bank': 1001}
Reviews inserted.
